# Convergence of α-D-IMF and neural α-CSBM

Adapted from [CSBM §4.1](https://arxiv.org/html/2502.01416v3#S4.SS1) and `csbm/notebooks/convergence_d_imf.ipynb`.

1. **Exact iteration:** compute the Markovian and reciprocal projections on a finite state space and compare relaxation parameters against one independently computed SB.
2. **Neural optimization:** compare simultaneous partial fitting (α-CSBM) with alternating fitting (CSBM), using the CSBM endpoint-prediction parameterization and AdamW. Evaluate both directions against the same full SB joint distribution.

The notebook is self-contained: it does not modify or import the project's model code and does not submit Slurm jobs. Run it in a notebook kernel with PyTorch, NumPy, pandas and Matplotlib. All experiment cells are initially unexecuted. CPU is sufficient; an allocated GPU can be selected for the neural part.

We use $D=1$, $S=50$, $p_0(a)=1/S$ and $p_1(b)=b/\sum_{s=1}^{S}s$. Factorization across coordinates is absent in this test. Neural fitting uses **exact finite-state expectations**, so this experiment isolates the parameterization and update schedule rather than minibatch noise. It does not test the high-dimensional factorization approximation.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

# The reference stochasticity is distinct from the relaxation parameter.
S = 50
N = 10                         # intermediate states: L=N+1 transitions
REFERENCE_NOISE = 0.01
REFERENCE_SKIP = 10             # base transitions per modeled interval
RELAXATIONS = [1.0, 0.5, 0.25, 0.1]
EXACT_ITERATIONS = 500
SINKHORN_TOL = 1e-12

SEEDS = [0, 1, 2]
HIDDEN_DIM = 128
EMBED_DIM = 32
LEARNING_RATE = 4e-4
WEIGHT_DECAY = 0.01
PRETRAIN_STEPS = 500            # joint steps on the independent coupling
MODEL_UPDATES = 6000            # one direction updated once counts as one
CSBM_FIT_STEPS = 250            # updates per fixed-target directional fit
EVAL_EVERY = 50                 # in directional model updates

QUICK_RUN = False               # debugging run, not publication results
if QUICK_RUN:
    EXACT_ITERATIONS = 30
    SEEDS = [0]
    PRETRAIN_STEPS = 10
    MODEL_UPDATES = 100
    CSBM_FIT_STEPS = 25
    EVAL_EVERY = 10

NEURAL_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64
# Tiny dense matrices are often faster with a small CPU thread count.
torch.set_num_threads(min(4, torch.get_num_threads()))
assert MODEL_UPDATES % 2 == 0 and EVAL_EVERY % 2 == 0

RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = Path.cwd() / 'outputs' / 'convergence_alpha_csbm' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
# Optional: set this to the article's figures/alpha-csbm directory before export.
PAPER_FIGURES_DIR = None

config = {
    'S': S, 'D': 1, 'N': N, 'reference': 'uniform',
    'reference_noise': REFERENCE_NOISE, 'reference_skip': REFERENCE_SKIP,
    'relaxations': RELAXATIONS, 'exact_iterations': EXACT_ITERATIONS,
    'seeds': SEEDS, 'hidden_dim': HIDDEN_DIM, 'embed_dim': EMBED_DIM,
    'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'pretrain_steps': PRETRAIN_STEPS, 'model_updates': MODEL_UPDATES,
    'csbm_fit_steps': CSBM_FIT_STEPS, 'eval_every': EVAL_EVERY,
    'quick_run': QUICK_RUN, 'neural_device': str(NEURAL_DEVICE),
    'dtype': str(DTYPE), 'torch_version': torch.__version__,
    'expectations': 'exact', 'optimizer': 'AdamW', 'betas': [0.95, 0.99],
}
(OUTPUT_DIR / 'config.json').write_text(json.dumps(config, indent=2))
print('Device:', NEURAL_DEVICE)
print('Outputs:', OUTPUT_DIR)

## 1. Reference and independently computed SB

The base uniform transition has diagonal $1-\varepsilon$ and off-diagonal $\varepsilon/(S-1)$. Each modeled interval contains `REFERENCE_SKIP` base transitions, matching the uniform construction used in the original notebook. The reference and time grid stay **fixed across relaxation parameters**.

Write $A_n=R_0\cdots R_{n-1}$, $H_n=R_n\cdots R_{L-1}$ and $K=A_L$. Sinkhorn scales $K$ to the endpoint constraints. Its residual and iteration count are recorded; the resulting SB is a numerical reference, not a closed-form solution.

In [ ]:
def log_prob(p):
    return p.clamp_min(torch.finfo(p.dtype).tiny).log()


def kl(p, q):
    """KL(p || q): a probability sum, without batch averaging."""
    value = torch.where(p > 0, p * (log_prob(p) - log_prob(q)), 0.0).sum()
    if value.item() < -1e-9:
        raise ArithmeticError(f'Negative KL: {value.item()}')
    return value.clamp_min(0)


def make_reference(s, n, noise, skip):
    assert 0 < noise < 1 and skip >= 1
    base = torch.full((s, s), noise / (s - 1), dtype=DTYPE)
    base.diagonal().fill_(1 - noise)
    interval = torch.linalg.matrix_power(base, skip)
    R = interval.repeat(n + 1, 1, 1)
    A = [torch.eye(s, dtype=DTYPE)]
    for matrix in R:
        A.append(A[-1] @ matrix)
    H = [None] * (n + 2)
    H[-1] = torch.eye(s, dtype=DTYPE)
    for j in reversed(range(n + 1)):
        H[j] = R[j] @ H[j + 1]
    A, H = torch.stack(A), torch.stack(H)
    assert (A[-1] > 0).all()
    return R, A, H


def sinkhorn_coupling(K, p0, p1, tol=1e-12, max_iterations=20000):
    logK = log_prob(K)
    logu, logv = torch.zeros_like(p0), torch.zeros_like(p1)
    for iteration in range(1, max_iterations + 1):
        logu = log_prob(p0) - torch.logsumexp(logK + logv[None, :], dim=1)
        logv = log_prob(p1) - torch.logsumexp(logK + logu[:, None], dim=0)
        if iteration % 10 == 0 or iteration == max_iterations:
            C = torch.exp(logu[:, None] + logK + logv[None, :])
            error = max((C.sum(1) - p0).abs().max().item(),
                        (C.sum(0) - p1).abs().max().item())
            if error < tol:
                return C, {'iterations': iteration, 'marginal_residual': error}
    raise RuntimeError(f'Sinkhorn did not reach tolerance: {error:.3g}')


R, A, H = make_reference(S, N, REFERENCE_NOISE, REFERENCE_SKIP)
L, K = N + 1, A[-1]
p0 = torch.full((S,), 1.0 / S, dtype=DTYPE)
p1 = torch.arange(1, S + 1, dtype=DTYPE)
p1 = p1 / p1.sum()
C0 = p0[:, None] * p1[None, :]
C_star, sinkhorn_info = sinkhorn_coupling(K, p0, p1, SINKHORN_TOL)
(OUTPUT_DIR / 'sinkhorn.json').write_text(json.dumps(sinkhorn_info, indent=2))
np.savez(OUTPUT_DIR / 'reference_and_sb.npz', R=R.numpy(), p0=p0.numpy(),
         p1=p1.numpy(), C_star=C_star.numpy())
print(sinkhorn_info)

## 2. Exact α-D-IMF

For an endpoint coupling $C$, the reciprocal joint distribution is $q_C=C(x_0,x_1)q^{\mathrm{ref}}(x_{\mathrm{in}}\mid x_0,x_1)$. Its neighboring-time joint distributions are

$$J_n=R_n\odot\left[A_n^\top(C/K)H_{n+1}^\top\right].$$

Normalizing $J_n$ gives the Markovian projection's transitions. Their product gives the next endpoint coupling. **Relax endpoint couplings, not transition matrices**:

$$C_\alpha^{k+1}=(1-\alpha)C_\alpha^k+\alpha\,[T(q_{C_\alpha^k})]_{0,1}.$$

Both $q_C$ and $q^\star$ have the same reference bridge distributions, so $\mathrm{KL}(q_C\|q^\star)=\mathrm{KL}(C\|C^\star)$. Each plotted iteration is one $T_\alpha$ application. Unlike the original notebook's forward/backward loop, it is not two projection cycles.

In [ ]:
def reciprocal_pairs(C, R, A, H):
    W = C / A[-1]
    return torch.stack([R[n] * (A[n].T @ W @ H[n + 1].T)
                        for n in range(R.shape[0])])


def transitions_from_pairs(J):
    forward = J / J.sum(dim=2, keepdim=True)
    backward = (J / J.sum(dim=1, keepdim=True)).transpose(1, 2)
    return forward, backward


def forward_coupling(F, initial):
    product = torch.eye(F.shape[1], dtype=F.dtype, device=F.device)
    for matrix in F:
        product = product @ matrix
    return initial[:, None] * product


def backward_coupling(B, terminal):
    product = torch.eye(B.shape[1], dtype=B.dtype, device=B.device)
    for matrix in reversed(B.unbind(0)):
        product = product @ matrix
    return (terminal[:, None] * product).T


def exact_step(C, relaxation, R, A, H, p0):
    J = reciprocal_pairs(C, R, A, H)
    F, _ = transitions_from_pairs(J)
    target = forward_coupling(F, p0)
    return (1 - relaxation) * C + relaxation * target


J_star = reciprocal_pairs(C_star, R, A, H)
F_star, B_star = transitions_from_pairs(J_star)
assert torch.allclose(forward_coupling(F_star, p0), C_star, atol=1e-10)
assert torch.allclose(backward_coupling(B_star, p1), C_star, atol=1e-10)

# Check the proposition's weighted Fisher--Rao step at one reciprocal iterate.
a_check = 0.3
J_old = reciprocal_pairs(C0, R, A, H)
F_old, _ = transitions_from_pairs(J_old)
C_target = forward_coupling(F_old, p0)
J_target = reciprocal_pairs(C_target, R, A, H)
J_new = (1 - a_check) * J_old + a_check * J_target
weights = J_new.sum(dim=2, keepdim=True)
gradient = (J_target.sum(dim=2, keepdim=True) * F_old - J_target) / weights
F_partial = F_old - a_check * gradient
F_new, _ = transitions_from_pairs(J_new)
assert torch.allclose(F_partial, F_new, atol=1e-12)
print('SB fixed point and one-step non-parametric identity: OK')

In [ ]:
exact_rows = []
exact_final = {}
for relaxation in RELAXATIONS:
    C = C0.clone()
    for iteration in range(EXACT_ITERATIONS + 1):
        exact_rows.append({'relaxation': relaxation, 'iteration': iteration,
                           'path_kl': kl(C, C_star).item(),
                           'marginal_error': max((C.sum(1) - p0).abs().max().item(),
                                                 (C.sum(0) - p1).abs().max().item())})
        if iteration < EXACT_ITERATIONS:
            C = exact_step(C, relaxation, R, A, H, p0)
    exact_final[str(relaxation)] = C.numpy()
exact_history = pd.DataFrame(exact_rows)
exact_history.to_csv(OUTPUT_DIR / 'alpha_dimf_history.csv', index=False)
np.savez(OUTPUT_DIR / 'alpha_dimf_final_couplings.npz', **exact_final)
print(exact_history.groupby('relaxation').tail(1).to_string(index=False))

In [ ]:
plt.rcParams.update({'font.size': 10, 'pdf.fonttype': 42, 'ps.fonttype': 42})
fig, ax = plt.subplots(figsize=(6.2, 3.5), layout='constrained')
for relaxation, group in exact_history.groupby('relaxation', sort=False):
    label = r'$\alpha=1$ (D-IMF)' if relaxation == 1 else fr'$\alpha={relaxation:g}$'
    ax.plot(group.iteration, group.path_kl.clip(lower=1e-14), label=label)
ax.set(xlabel=r'$T_\alpha$ iterations ($k$)', ylabel=r'$\mathrm{KL}(q_\alpha^k\|q^\star)$', yscale='log')
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
fig.savefig(OUTPUT_DIR / 'alpha_dimf_convergence.pdf')
fig.savefig(OUTPUT_DIR / 'alpha_dimf_convergence.png', dpi=220)
plt.show()
# Only the display is floored at 1e-14; the CSV retains measured values.

## 3. Neural transition models and full-joint evaluation

Each direction predicts an endpoint distribution conditioned on the current state and time, then mixes the corresponding reference bridge transitions, as in the CSBM parameterization. The two models have separate parameters. We use a small MLP with three hidden layers of width `HIDDEN_DIM`, state/time embeddings and softmax endpoint probabilities.

Forward bridge transitions are $R_n(x,y)H_{n+1}(y,b)/H_n(x,b)$. Backward bridge transitions are $A_n(a,x)R_n(x,y)/A_{n+1}(a,y)$. The latter are stored with rows indexed by the current state $y$, columns by the preceding state $x$.

For each evaluated model, full-joint KL uses **that model's own time marginals** in the transition-chain decomposition. Endpoint KL and endpoint marginal errors are also logged as diagnostics; they do not replace full-joint KL.

In [ ]:
def bridge_transitions(R, A, H):
    # Forward dimensions: n, current x, next y, endpoint b.
    BF = torch.stack([R[n, :, :, None] * H[n + 1][None, :, :]
                      / H[n][:, None, :] for n in range(R.shape[0])])
    # Backward dimensions before permutation: n, endpoint a, previous x, current y.
    BB = torch.stack([A[n, :, :, None] * R[n][None, :, :]
                      / A[n + 1][:, None, :] for n in range(R.shape[0])])
    BB = BB.permute(0, 3, 2, 1).contiguous()
    assert torch.allclose(BF.sum(2), torch.ones_like(BF.sum(2)), atol=1e-12)
    assert torch.allclose(BB.sum(2), torch.ones_like(BB.sum(2)), atol=1e-12)
    return BF, BB


class EndpointPredictor(nn.Module):
    def __init__(self, s, length, hidden, embedding):
        super().__init__()
        self.state_embedding = nn.Embedding(s, embedding)
        self.time_embedding = nn.Embedding(length, embedding)
        self.network = nn.Sequential(
            nn.Linear(2 * embedding, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, s))
        self.register_buffer('states', torch.arange(s).repeat(length))
        self.register_buffer('times', torch.arange(length).repeat_interleave(s))
        self.length, self.s = length, s

    def forward(self):
        features = torch.cat([self.state_embedding(self.states),
                              self.time_embedding(self.times)], dim=-1)
        return self.network(features).reshape(self.length, self.s, self.s).softmax(-1)


def model_transitions(model, bridges):
    # All states/times are enumerated: no transition sampling approximation.
    return torch.einsum('nxe,nxye->nxy', model(), bridges)


def path_kl(transitions, initial, star_transitions, star_initial, backward=False):
    marginal = initial
    value = kl(initial, star_initial)
    indices = reversed(range(transitions.shape[0])) if backward else range(transitions.shape[0])
    for n in indices:
        row_kl = torch.where(transitions[n] > 0,
                            transitions[n] * (log_prob(transitions[n]) - log_prob(star_transitions[n])),
                            0.0).sum(dim=1)
        value = value + (marginal * row_kl).sum()
        marginal = marginal @ transitions[n]
    if value.item() < -1e-9:
        raise ArithmeticError(f'Negative full-joint KL: {value.item()}')
    return value.clamp_min(0)


def fitting_loss(transitions, target_pairs, backward=False):
    # Cross-entropy is KL(target || model) up to a parameter-independent term.
    # Division by L only fixes the learning-rate convention.
    weights = target_pairs.transpose(1, 2) if backward else target_pairs
    return -(weights * log_prob(transitions)).sum() / transitions.shape[0]


BF, BB = bridge_transitions(R, A, H)
assert path_kl(F_star, p0, F_star, p0).item() < 1e-12
assert path_kl(B_star, p1, B_star, p1, backward=True).item() < 1e-12

data = {name: tensor.to(NEURAL_DEVICE) for name, tensor in {
    'R': R, 'A': A, 'H': H, 'p0': p0, 'p1': p1,
    'C0': C0, 'C_star': C_star, 'F_star': F_star, 'B_star': B_star,
    'BF': BF, 'BB': BB,
}.items()}

## 4. Common initialization and update schedules

Both methods start from the **same pretrained weights and AdamW state** for each seed. Pretraining fits both directions to the independent coupling's reference bridges.

- **α-CSBM:** construct both reciprocal targets from the current opposite models, detach them, and update both models once. Targets are constructed before either model changes.
- **CSBM:** fix the reciprocal target from the opposite model and update one direction `CSBM_FIT_STEPS` times, then switch direction. This finite fitting budget is a practical alternating baseline, not an exact Markovian projection.

The learning rate of AdamW is not identified with the theoretical relaxation parameter. Only the exact experiment sweeps $\alpha$.

For the comparison budget, **one update of one direction counts as one model update**. A joint α-CSBM optimizer step counts as two. The x-axis excludes the shared pretraining, whose history and state are saved separately. Each inner AdamW update in the alternating baseline counts as one.

In [ ]:
def new_bundle(seed):
    torch.manual_seed(seed)
    if NEURAL_DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)
    f = EndpointPredictor(S, L, HIDDEN_DIM, EMBED_DIM).to(device=NEURAL_DEVICE, dtype=DTYPE)
    b = EndpointPredictor(S, L, HIDDEN_DIM, EMBED_DIM).to(device=NEURAL_DEVICE, dtype=DTYPE)
    optimizer = torch.optim.AdamW(list(f.parameters()) + list(b.parameters()),
                                 lr=LEARNING_RATE, betas=(0.95, 0.99),
                                 weight_decay=WEIGHT_DECAY)
    return {'forward': f, 'backward': b, 'optimizer': optimizer}


def sync_device():
    if NEURAL_DEVICE.type == 'cuda':
        torch.cuda.synchronize()


@torch.no_grad()
def evaluate_bundle(bundle, method, seed, updates, elapsed):
    F = model_transitions(bundle['forward'], data['BF'])
    B = model_transitions(bundle['backward'], data['BB'])
    Cf = forward_coupling(F, data['p0'])
    Cb = backward_coupling(B, data['p1'])
    return {
        'method': method, 'seed': seed, 'model_updates': updates,
        'elapsed_seconds': elapsed,
        'forward_path_kl': path_kl(F, data['p0'], data['F_star'], data['p0']).item(),
        'backward_path_kl': path_kl(B, data['p1'], data['B_star'], data['p1'], backward=True).item(),
        'forward_endpoint_kl': kl(Cf, data['C_star']).item(),
        'backward_endpoint_kl': kl(Cb, data['C_star']).item(),
        'forward_target_tv': 0.5 * (Cf.sum(0) - data['p1']).abs().sum().item(),
        'backward_target_tv': 0.5 * (Cb.sum(1) - data['p0']).abs().sum().item(),
    }


def bundle_state(bundle):
    return {name: copy.deepcopy(item.state_dict()) for name, item in bundle.items()}


def restore_bundle(seed, state):
    bundle = new_bundle(seed)
    for name, item in bundle.items():
        item.load_state_dict(copy.deepcopy(state[name]))
    return bundle


def pretrain(seed):
    bundle = new_bundle(seed)
    J0 = reciprocal_pairs(data['C0'], data['R'], data['A'], data['H'])
    history = [evaluate_bundle(bundle, 'pretraining', seed, 0, 0.0)]
    sync_device()
    start = time.perf_counter()
    for step in range(1, PRETRAIN_STEPS + 1):
        bundle['optimizer'].zero_grad(set_to_none=True)
        F = model_transitions(bundle['forward'], data['BF'])
        B = model_transitions(bundle['backward'], data['BB'])
        loss = 0.5 * (fitting_loss(F, J0) + fitting_loss(B, J0, backward=True))
        loss.backward()
        bundle['optimizer'].step()
        if 2 * step % EVAL_EVERY == 0 or step == PRETRAIN_STEPS:
            sync_device()
            history.append(evaluate_bundle(bundle, 'pretraining', seed, 2 * step,
                                           time.perf_counter() - start))
    state = bundle_state(bundle)
    torch.save(state, OUTPUT_DIR / f'pretrained_seed_{seed}.pt')
    pd.DataFrame(history).to_csv(OUTPUT_DIR / f'pretraining_seed_{seed}.csv', index=False)
    return state


def run_online(seed, initial_state):
    bundle = restore_bundle(seed, initial_state)
    rows = [evaluate_bundle(bundle, 'alpha-CSBM', seed, 0, 0.0)]
    sync_device()
    start = time.perf_counter()
    for updates in range(2, MODEL_UPDATES + 1, 2):
        with torch.no_grad():
            F = model_transitions(bundle['forward'], data['BF'])
            B = model_transitions(bundle['backward'], data['BB'])
            Cf = forward_coupling(F, data['p0'])
            Cb = backward_coupling(B, data['p1'])
            target_f = reciprocal_pairs(Cb, data['R'], data['A'], data['H'])
            target_b = reciprocal_pairs(Cf, data['R'], data['A'], data['H'])
        bundle['optimizer'].zero_grad(set_to_none=True)
        F = model_transitions(bundle['forward'], data['BF'])
        B = model_transitions(bundle['backward'], data['BB'])
        loss = 0.5 * (fitting_loss(F, target_f) + fitting_loss(B, target_b, backward=True))
        loss.backward()
        bundle['optimizer'].step()
        if updates % EVAL_EVERY == 0 or updates == MODEL_UPDATES:
            sync_device()
            rows.append(evaluate_bundle(bundle, 'alpha-CSBM', seed, updates,
                                        time.perf_counter() - start))
    return bundle, rows


def run_alternating(seed, initial_state):
    bundle = restore_bundle(seed, initial_state)
    rows = [evaluate_bundle(bundle, 'CSBM', seed, 0, 0.0)]
    updates, block = 0, 0
    sync_device()
    start = time.perf_counter()
    while updates < MODEL_UPDATES:
        direction = 'forward' if block % 2 == 0 else 'backward'
        with torch.no_grad():
            if direction == 'forward':
                other = model_transitions(bundle['backward'], data['BB'])
                C_target = backward_coupling(other, data['p1'])
            else:
                other = model_transitions(bundle['forward'], data['BF'])
                C_target = forward_coupling(other, data['p0'])
            target = reciprocal_pairs(C_target, data['R'], data['A'], data['H'])
        for _ in range(min(CSBM_FIT_STEPS, MODEL_UPDATES - updates)):
            bundle['optimizer'].zero_grad(set_to_none=True)
            bridges = data['BF'] if direction == 'forward' else data['BB']
            transitions = model_transitions(bundle[direction], bridges)
            loss = fitting_loss(transitions, target, backward=(direction == 'backward'))
            # Match the 1/2 normalization of the joint online loss for each direction.
            (0.5 * loss).backward()
            bundle['optimizer'].step()
            updates += 1
            if updates % EVAL_EVERY == 0 or updates == MODEL_UPDATES:
                sync_device()
                rows.append(evaluate_bundle(bundle, 'CSBM', seed, updates,
                                            time.perf_counter() - start))
        block += 1
    return bundle, rows

In [ ]:
neural_rows = []
for seed in SEEDS:
    initial_state = pretrain(seed)
    for method, runner in [('alpha-CSBM', run_online), ('CSBM', run_alternating)]:
        bundle, rows = runner(seed, initial_state)
        neural_rows.extend(rows)
        torch.save(bundle_state(bundle), OUTPUT_DIR / f'{method}_seed_{seed}_final.pt')
        pd.DataFrame(rows).to_csv(OUTPUT_DIR / f'{method}_seed_{seed}_history.csv', index=False)
        print(method, 'seed', seed, 'final full-joint KL:',
              rows[-1]['forward_path_kl'], rows[-1]['backward_path_kl'])
        del bundle
    del initial_state
neural_history = pd.DataFrame(neural_rows)
neural_history.to_csv(OUTPUT_DIR / 'neural_history.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.5), layout='constrained')
colors = {'alpha-CSBM': '#0072B2', 'CSBM': '#D55E00'}
summary_rows = []
for method in ['alpha-CSBM', 'CSBM']:
    for direction, style in [('forward', '-'), ('backward', '--')]:
        column = f'{direction}_path_kl'
        grouped = neural_history[neural_history.method == method].groupby('model_updates')[column]
        stats = grouped.agg(['mean', 'std', 'count']).reset_index()
        stats['std'] = stats['std'].fillna(0)
        ax.plot(stats.model_updates, stats['mean'].clip(lower=1e-14),
                color=colors[method], linestyle=style, label=f'{method}, {direction}')
        ax.fill_between(stats.model_updates, (stats['mean'] - stats['std']).clip(lower=1e-14),
                        (stats['mean'] + stats['std']).clip(lower=1e-14),
                        color=colors[method], alpha=0.10)
        stats['method'], stats['direction'] = method, direction
        summary_rows.append(stats)
ax.set(xlabel='Directional model updates after common pretraining',
       ylabel=r'$\mathrm{KL}(m_\theta\|q^\star)$', yscale='log')
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.savefig(OUTPUT_DIR / 'alpha_csbm_convergence.pdf')
fig.savefig(OUTPUT_DIR / 'alpha_csbm_convergence.png', dpi=220)
plt.show()
summary = pd.concat(summary_rows, ignore_index=True)
summary.to_csv(OUTPUT_DIR / 'neural_summary.csv', index=False)
print('Lines: seed means; shading: ±1 sample standard deviation (not a confidence interval).')
print(neural_history.sort_values('model_updates').groupby(['method', 'seed']).tail(1)[
    ['method', 'seed', 'model_updates', 'forward_path_kl', 'backward_path_kl',
     'forward_target_tv', 'backward_target_tv']].to_string(index=False))

## 5. Export to the manuscript

The main figures are `alpha_dimf_convergence.pdf` and `alpha_csbm_convergence.pdf`. Their source values, Sinkhorn residual, configuration and neural states are saved beside them. Set `PAPER_FIGURES_DIR` to the article's `figures/alpha-csbm` directory to copy the two figures explicitly. The notebook does not rewrite LaTeX or fill in claims about unobserved results.

In `sections/experiments-convergence.tex`, replace each dashed placeholder with its commented `\includegraphics` line after reviewing the figures. Keep the measured error floor and the finite alternating fitting budget visible when interpreting the results. A small finite-state experiment checks convergence in this setup; it is not a universal convergence guarantee for AdamW or the factorized CelebA model.

In [ ]:
if PAPER_FIGURES_DIR is not None:
    import shutil
    destination = Path(PAPER_FIGURES_DIR).expanduser()
    destination.mkdir(parents=True, exist_ok=True)
    for name in ['alpha_dimf_convergence.pdf', 'alpha_csbm_convergence.pdf']:
        shutil.copy2(OUTPUT_DIR / name, destination / name)
    print('Copied figures to:', destination)
else:
    print('Figures remain in:', OUTPUT_DIR)